#### Imports

In [ ]:
import h5py
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import zscore
from scipy.signal import decimate

#### Downsampling, normalization, and sliding window

In [ ]:
folder_path_intra = r"G:\Coursework\Period4\DL\Assignment2\data\Intra"
folder_path_cross = r"G:\Coursework\Period4\DL\Assignment2\data\Cross"

DOWNSAMPLE_FACTOR = 10
# After 10x downsampling, ~2035 Hz -> ~200 Hz
# Window of 256 = ~1.28 seconds; step of 64 = 75% overlap
WINDOW_SIZE = 256
STEP_SIZE = 64

def get_label(filename):
    if "rest" in filename:
        return 0
    elif "motor" in filename:
        return 1
    elif "working_memory" in filename:
        return 2
    elif "story" in filename:
        return 3
    return -1

def sliding_window(data, window_size, step_size):
    """
    Slice a single trial (channels x time) into overlapping windows.
    Returns array of shape (n_windows, channels, window_size).
    """
    n_channels, n_times = data.shape
    windows = []
    for start in range(0, n_times - window_size + 1, step_size):
        windows.append(data[:, start:start + window_size])
    return np.array(windows)  # (n_windows, channels, window_size)

def load_folder(folder_path, window_size=WINDOW_SIZE, step_size=STEP_SIZE):
    X, y = [], []

    for file in sorted(os.listdir(folder_path)):
        if not file.endswith(".h5"):
            continue

        label = get_label(file)
        if label == -1:
            continue

        file_path = os.path.join(folder_path, file)
        with h5py.File(file_path, 'r') as f:
            key = list(f.keys())[0]
            data = f[key][:]  # (channels, time)

        # Downsample along time axis
        data = decimate(data, DOWNSAMPLE_FACTOR, axis=1)
        # Z-score per channel
        data = zscore(data, axis=1)

        # Slice into windows -> (n_windows, channels, window_size)
        windows = sliding_window(data, window_size, step_size)
        X.append(windows)
        y.extend([label] * len(windows))

    return np.concatenate(X, axis=0), np.array(y)


X_intra_train, y_intra_train = load_folder(folder_path_intra + "/train")
X_intra_test, y_intra_test = load_folder(folder_path_intra + "/test")

X_cross_train, y_cross_train = load_folder(folder_path_cross + "/train")
X_cross_test1, y_cross_test1 = load_folder(folder_path_cross + "/test1")
X_cross_test2, y_cross_test2 = load_folder(folder_path_cross + "/test2")
X_cross_test3, y_cross_test3 = load_folder(folder_path_cross + "/test3")
X_cross_test = np.concatenate([X_cross_test1, X_cross_test2, X_cross_test3], axis=0)
y_cross_test = np.concatenate([y_cross_test1, y_cross_test2, y_cross_test3], axis=0)

print("INTRA train:", X_intra_train.shape, y_intra_train.shape)
print("INTRA test: ", X_intra_test.shape,  y_intra_test.shape)
print("CROSS train:", X_cross_train.shape, y_cross_train.shape)
print("CROSS test: ", X_cross_test.shape,  y_cross_test.shape)

#### EEGNet (sized for windowed input)

In [ ]:
class EEGNet(nn.Module):
    def __init__(self, n_classes=4, n_channels=248, n_timesteps=256, F1=8, D=2, F2=16, dropout_rate=0.5):
        super(EEGNet, self).__init__()

        # Block 1: Temporal conv
        # Kernel = half the sampling rate (~100 at 200 Hz) as per original EEGNet paper
        self.temp_conv = nn.Sequential(
            nn.Conv2d(1, F1, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1)
        )

        # Block 2: Depthwise conv across channels
        self.depth_conv = nn.Sequential(
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout_rate)
        )

        # Block 3: Separable conv
        self.sep_conv = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout_rate)
        )

        # Compute flattened size dynamically to avoid mismatches
        # After pool1 (÷4) and pool2 (÷8): timesteps // 32
        flat_size = F2 * (n_timesteps // 32)

        # Classification head with an extra hidden layer for regularization
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 64),
            nn.ELU(),
            nn.Dropout(0.5),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        x = self.temp_conv(x)
        x = self.depth_conv(x)
        x = self.sep_conv(x)
        x = self.classifier(x)
        return x

#### Training and evaluation

In [ ]:
def make_loader(X, y, batch_size=32, shuffle=True):
    X_tensor = torch.FloatTensor(X).unsqueeze(1)  # (N, 1, channels, time)
    y_tensor = torch.LongTensor(y)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=shuffle)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct = 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            out = model(X_batch)
            total_loss += criterion(out, y_batch).item()
            correct += (out.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def train_with_early_stopping(model, train_loader, test_loader, criterion, optimizer,
                               scheduler, device, patience=15, epochs=100):
    best_test_loss = float('inf')
    best_weights = None
    counter = 0

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        scheduler.step(test_loss)

        print(f"Epoch {epoch+1:3d}: Train Loss={train_loss:.4f} Acc={train_acc:.4f} | "
              f"Test Loss={test_loss:.4f} Acc={test_acc:.4f}")

        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            counter = 0
        else:
            counter += 1
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_weights)
    return model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

# ── Intra-subject ──────────────────────────────────────────────────────────────
print("--- Intra Subject Training ---")
model_intra = EEGNet(n_classes=4, n_channels=248, n_timesteps=WINDOW_SIZE,
                     dropout_rate=0.5).to(device)
optimizer_intra = optim.Adam(model_intra.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_intra = optim.lr_scheduler.ReduceLROnPlateau(optimizer_intra, patience=5, factor=0.5, verbose=True)

train_loader_intra = make_loader(X_intra_train, y_intra_train, batch_size=64)
test_loader_intra  = make_loader(X_intra_test,  y_intra_test,  batch_size=64, shuffle=False)

model_intra = train_with_early_stopping(
    model_intra, train_loader_intra, test_loader_intra,
    criterion, optimizer_intra, scheduler_intra, device
)

# ── Cross-subject ──────────────────────────────────────────────────────────────
print("\n--- Cross Subject Training ---")
model_cross = EEGNet(n_classes=4, n_channels=248, n_timesteps=WINDOW_SIZE,
                     dropout_rate=0.5).to(device)
optimizer_cross = optim.Adam(model_cross.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_cross = optim.lr_scheduler.ReduceLROnPlateau(optimizer_cross, patience=5, factor=0.5, verbose=True)

train_loader_cross = make_loader(X_cross_train, y_cross_train, batch_size=64)
test_loader_cross  = make_loader(X_cross_test,  y_cross_test,  batch_size=64, shuffle=False)

model_cross = train_with_early_stopping(
    model_cross, train_loader_cross, test_loader_cross,
    criterion, optimizer_cross, scheduler_cross, device
)